# Problem 7 — PyTorch Modules and a Fresh ResNet Cut

Use the exact My_CamelCase module identifiers and the course's float32 pretrained-model register.


In [ ]:
import os
import pathlib

_root = next(
    path for path in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
from torch import nn
from torchvision.models import ResNet50_Weights, resnet50


## Part 7.1 (5 points)

**Type:** programming · **Difficulty: core** · **Answer form: code** · **Concepts: nn-module, custom-layers**  
**Flag:** Coding allowed.

Complete the exact module class Quay_Affine. Its constructor accepts a rank-2 float64 weight and rank-1 float64 bias, registers independent copies as nn.Parameter objects with requires_grad=False, and forward(x) returns x @ weight.T + bias. For weight shape (out_features, in_features), support any input x with final dimension in_features.


In [ ]:
class Quay_Affine(nn.Module):
    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(weight.detach().clone(), requires_grad=False)
        self.bias = nn.Parameter(bias.detach().clone(), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias


_shape_layer = Quay_Affine(torch.zeros(4, 3, dtype=torch.float64), torch.zeros(4, dtype=torch.float64))
assert dict(_shape_layer.named_parameters()).keys() == {"weight", "bias"}
assert all(not p.requires_grad for p in _shape_layer.parameters())

In [ ]:
registered_scalar_count = sum(parameter.numel() for parameter in _shape_layer.parameters())
ANSWER = 16
assert torch.isclose(torch.tensor(float(ANSWER)), torch.tensor(float(registered_scalar_count)), atol=0, rtol=0)

## Part 7.2 (5 points)

**Type:** programming · **Difficulty: core** · **Answer form: code** · **Concepts: torch-tensors, manual-weights**  
**Flag:** Coding allowed.

Using Quay_Affine from Part 7.1, hand-set float64 tensors manual_weight and manual_bias so manual_layer maps each (x1, x2, x3) to

(2x1 - x2 + 3x3 + 1, -4x1 + 2x2 - x3 - 2, x1 + x2 - 2x3 + 0.5, 3x2 + x3 - 1).

Run the supplied batch and keep manual_output. The asserted shape contract is required.


In [ ]:
manual_weight = torch.tensor([
    [2.0, -1.0, 3.0],
    [-4.0, 2.0, -1.0],
    [1.0, 1.0, -2.0],
    [0.0, 3.0, 1.0],
], dtype=torch.float64)
manual_bias = torch.tensor([1.0, -2.0, 0.5, -1.0], dtype=torch.float64)
manual_layer = Quay_Affine(manual_weight, manual_bias)

manual_input = torch.tensor([
    [1.0, -2.0, 0.5],
    [-3.0, 1.0, 2.0],
], dtype=torch.float64)
manual_output = manual_layer(manual_input)
assert manual_output.shape == (2, 4)
assert manual_output.dtype == torch.float64

In [ ]:
ANSWER = -3.5
assert torch.isclose(torch.tensor(ANSWER, dtype=manual_output.dtype), manual_output.sum(), atol=1e-12, rtol=0)

## Part 7.3 (5 points)

**Type:** theory · **Difficulty: advanced** · **Answer form: numeric** · **Concepts: bottleneck-blocks, parameter-counting**  
**Flags:** Reasoning required; coding not allowed.  
**Ban (zero points for this part):** Do not use numel, nelement, parameter-iteration counts, or any disguised counting helper.

A fresh interior bottleneck has width m = 88 and no downsample. Its three bias-free convolutions map 352 to 88 with a 1 by 1 kernel, 88 to 88 with a 3 by 3 kernel, and 88 to 352 with a 1 by 1 kernel. Count only convolution weights, not BatchNorm parameters. Give the exact integer count and show the three product terms.


**Solution.** The three convolution-weight counts are

$$88\cdot352\cdot1\cdot1 + 88\cdot88\cdot3\cdot3
+ 352\cdot88\cdot1\cdot1
=30{,}976+69{,}696+30{,}976=131{,}648.$$

Thus the exact count, excluding BatchNorm, is **131648**.

In [ ]:
convolution_weight_count = 88 * 352 + 88 * 88 * 3 * 3 + 352 * 88
ANSWER = 131648
assert torch.isclose(torch.tensor(float(ANSWER)), torch.tensor(float(convolution_weight_count)), atol=0, rtol=0)

## Part 7.4 (5 points)

**Type:** programming · **Difficulty: advanced** · **Answer form: code** · **Concepts: model-truncation, resnet-architecture**  
**Flag:** Coding allowed.

Load the locally cached ResNet-50 with ResNet50_Weights.IMAGENET1K_V1 and immediately call eval(). Build resnet_cut as a sequential backbone through all of layer2 plus exactly the first two blocks of layer3, and put the new sequential module in eval mode too. This cut is specified as the first six top-level children followed by resnet.layer3[:2]. Run the supplied float32 input only inside torch.inference_mode(), store cut_features, and satisfy the output-shape assertion.


In [ ]:
resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
resnet.eval()
resnet_cut = nn.Sequential(*list(resnet.children())[:6], resnet.layer3[:2])
resnet_cut.eval()

torch.manual_seed(20260874)
image_batch = torch.randn(1, 3, 160, 160, dtype=torch.float32)
with torch.inference_mode():
    cut_features = resnet_cut(image_batch)

assert not resnet.training
assert not resnet_cut.training
assert cut_features.dtype == torch.float32
assert cut_features.shape == (1, 1024, 10, 10)

In [ ]:
ANSWER = 1024
assert torch.isclose(torch.tensor(float(ANSWER)), torch.tensor(float(cut_features.shape[1])), atol=0, rtol=0)

## Part 7.5 (5 points)

**Type:** programming · **Difficulty: core** · **Answer form: code** · **Concepts: layer-freezing, requires-grad**  
**Flag:** Coding allowed.

Consume resnet_cut from Part 7.4 in the supplied Cut_Classifier, which adds adaptive pooling and a fresh 11-class trainable head. Freeze every backbone parameter without freezing the head. Store the scalar totals in frozen_scalars and trainable_scalars, counting from classifier.named_parameters(), and keep the supplied audit assertions.


In [ ]:
class Cut_Classifier(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(1024, 11)

    def forward(self, x):
        features = self.backbone(x)
        return self.head(torch.flatten(self.pool(features), 1))


classifier = Cut_Classifier(resnet_cut)
for parameter in classifier.backbone.parameters():
    parameter.requires_grad_(False)

frozen_scalars = sum(
    parameter.numel()
    for name, parameter in classifier.named_parameters()
    if name.startswith("backbone.") and not parameter.requires_grad
)
trainable_scalars = sum(
    parameter.numel()
    for name, parameter in classifier.named_parameters()
    if parameter.requires_grad
)

assert all(not p.requires_grad for p in classifier.backbone.parameters())
assert all(p.requires_grad for p in classifier.head.parameters())
assert frozen_scalars > 0
assert trainable_scalars == 1024 * 11 + 11

In [ ]:
ANSWER = 4074560
assert torch.isclose(torch.tensor(float(ANSWER)), torch.tensor(float(frozen_scalars)), atol=0, rtol=0)